# LEDGAR Evaluation Metrics  — NVIDIA GPU copy

> **GPU-ready copy.** Originals live in the parent folder and are untouched.
> Pure metrics tutorial (no training) — the macro-F1 discipline the eval story rests on.
> Detects `cuda` automatically. Run the install cell first on a fresh cluster node.

In [ ]:
# --- Run once on the NVIDIA cluster to install dependencies ---
# NOTE: on the TR H100 cluster, install torch with the correct CUDA wheel FIRST, e.g.:
#   pip install torch --index-url https://download.pytorch.org/whl/cu124
# then the rest:
!pip install -q "transformers>=4.44" "datasets>=2.19,<3" scikit-learn accelerate
print("deps installed")

# Evaluating a classifier on LEDGAR — the metrics, explained from scratch

**No model in this notebook.** Before we ever fine-tune, we need to understand *how we'll
grade the model* — otherwise we can't tell whether training helped. This notebook teaches the
evaluation metrics for LEDGAR using **tiny, hand-made examples** you can trace by hand.

Why this matters so much: LEDGAR has **100 labels and they're very imbalanced** (some classes
have thousands of examples, some have ~20). On imbalanced data, the "obvious" metric
(**accuracy**) is *misleading* — a lazy model can score high while being useless. Choosing the
right metric is the difference between fooling yourself and actually measuring progress.

What you'll understand by the end:
1. What a "prediction vs truth" comparison looks like.
2. **Accuracy** — and a live demo of *why it lies* on imbalanced data.
3. **Precision, Recall, F1** — per class, in plain words.
4. **Macro vs Micro vs Weighted F1** — which one to report for LEDGAR, and why.
5. A clear rule for **which number to watch during training**.

> **Kernel:** `cuad-finetune (.venv)` (top-right) — just the environment name; it has what we need.


## Part 0 — What does "evaluating" even mean?

Evaluation = **compare the model's guesses to the correct answers, and turn that into a
number.** For classification it's beautifully simple. For each clause we have:

- **gold** (a.k.a. "truth" / "label") — the correct class, from the dataset.
- **pred** — the class the model guessed.

If `pred == gold`, that clause was classified correctly; otherwise wrong. A **metric** is
just a recipe for summarizing many of these right/wrong outcomes into one score.

Everything below is different recipes for that summary — and *when each recipe misleads you.*

We'll use a tiny made-up example so the arithmetic is visible. Imagine only **3 classes**
(in reality LEDGAR has 100) and **10 clauses**:

- class `0` = Governing Laws   (common)
- class `1` = Termination      (rarer)
- class `2` = Confidentiality  (rarer)


In [ ]:
# gold = the correct label for each of 10 clauses (deliberately imbalanced: mostly class 0)
gold = [0, 0, 0, 0, 0, 0, 1, 1, 2, 2]

# pred = what some model guessed. Notice it LOVES class 0 (over-predicts the common class).
pred = [0, 0, 0, 0, 0, 0, 0, 1, 0, 2]

names = {0: "Governing Laws", 1: "Termination", 2: "Confidentiality"}

print("idx | gold             | pred             | correct?")
print("----+------------------+------------------+---------")
for i,(g,p) in enumerate(zip(gold,pred)):
    print(f"{i:3d} | {names[g]:16s} | {names[p]:16s} | {'yes' if g==p else 'NO'}")

Look at the table: the model got **8 of 10** right. The 2 mistakes are both cases where a
*rare* clause (Termination at idx 6, Confidentiality at idx 8) was wrongly called *Governing
Laws*. Hold that thought — it's the whole story.

## Part 1 — Accuracy (the obvious metric... that lies)

**Accuracy = (number correct) / (total).** Simple and intuitive. Let's compute it by hand,
then confirm with the standard library.

In [ ]:
correct = sum(1 for g, p in zip(gold, pred) if g == p)
total   = len(gold)
accuracy = correct / total
print(f"By hand: {correct} correct / {total} total = accuracy {accuracy:.2f}")

# The library version (scikit-learn) — this is what you'll actually call later:
from sklearn.metrics import accuracy_score
print("sklearn accuracy_score       :", accuracy_score(gold, pred))

80% — sounds good! Now watch the trap. Consider a **useless model that always guesses the
most common class** (class 0, Governing Laws), no matter what the clause says. It "understands"
nothing. What accuracy does it get?

In [ ]:
dumb = [0] * len(gold)          # always predicts class 0, ignoring the input entirely
print("Dumb model's predictions:", dumb)
print("Dumb model accuracy      :", accuracy_score(gold, dumb))

**60% accuracy for a model that literally ignores the input.** That's the trap.

Because 6 of our 10 clauses happen to be class 0, "always say class 0" is right 60% of the
time. On real LEDGAR, where the biggest class dwarfs the smallest ~137x, this effect is even
more extreme — a do-nothing model can post a *deceptively high* accuracy.

> **Lesson 1:** On imbalanced data, **accuracy rewards laziness**. A high accuracy might just
> mean "the model learned to always guess the popular classes." We need a metric that checks
> whether the model handles *every* class — including the rare ones.

That metric is built from **precision** and **recall**.

## Part 2 — Precision & Recall (per class)

These two always come as a pair, and they're defined **for one class at a time**. Pick a class
(say Termination) and ask two different questions:

- **Precision** — *"When the model said 'Termination', how often was it right?"*
  = correct Termination predictions ÷ **all** times it predicted Termination.
  (Measures: are the model's *claims* trustworthy? Low precision = many false alarms.)

- **Recall** — *"Of all the real Termination clauses, how many did the model catch?"*
  = correct Termination predictions ÷ **all** actual Termination clauses.
  (Measures: does the model *miss* things? Low recall = many misses.)

A memory hook:
- Precision = *"of what I flagged, how much was correct"* (guards against false alarms).
- Recall = *"of what exists, how much I found"* (guards against misses).

Let's compute both, for every class, by hand.

In [ ]:
def precision_recall_for(cls):
    tp = sum(1 for g, p in zip(gold, pred) if p == cls and g == cls)   # predicted cls AND correct
    predicted_cls = sum(1 for p in pred if p == cls)                    # everything predicted as cls
    actual_cls    = sum(1 for g in gold if g == cls)                    # everything truly cls
    precision = tp / predicted_cls if predicted_cls else 0.0
    recall    = tp / actual_cls if actual_cls else 0.0
    return tp, predicted_cls, actual_cls, precision, recall

print(f"{'class':16s} {'TP':>3} {'#pred':>6} {'#actual':>8} {'precision':>10} {'recall':>8}")
for cls in [0, 1, 2]:
    tp, npred, nact, prec, rec = precision_recall_for(cls)
    print(f"{names[cls]:16s} {tp:3d} {npred:6d} {nact:8d} {prec:10.3f} {rec:8.3f}")

Read the Governing Laws (class 0) row carefully — this is where the lazy behavior shows up:

- **Recall = 1.000**: it caught *every* real Governing Laws clause (of course — it over-predicts it).
- **Precision = 0.750**: but only 3/4 of the things it *called* Governing Laws actually were —
  because it wrongly grabbed a Termination and a Confidentiality clause too.

And the rare classes (Termination, Confidentiality) have **recall = 0.5** — the model *missed
half of them*. Accuracy hid this completely; precision/recall expose it. This is exactly the
per-class insight we needed.

## Part 3 — F1: combining precision & recall into one number

Often you want a **single** number per class, not two. **F1** is the standard way: it's the
*harmonic mean* of precision and recall. The key property of the harmonic mean is that it's
**only high when BOTH are high** — you can't fake it by acing one and tanking the other.

    F1 = 2 * (precision * recall) / (precision + recall)

Let's compute per-class F1 by hand, then confirm with the library.

In [ ]:
def f1_for(cls):
    *_, prec, rec = precision_recall_for(cls)
    if prec + rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)

print(f"{'class':16s} {'F1':>7}")
for cls in [0, 1, 2]:
    print(f"{names[cls]:16s} {f1_for(cls):7.3f}")

# Library check — per-class precision/recall/f1 in one call:
from sklearn.metrics import precision_recall_fscore_support
p, r, f, support = precision_recall_fscore_support(gold, pred, labels=[0,1,2], zero_division=0)
print("\nsklearn per-class F1:", [round(x,3) for x in f])
print("sklearn support (n) :", list(support))

So each class gets its own F1: Governing Laws ≈ 0.857, and the two rare classes ≈ 0.667.
Now — how do we roll 100 per-class F1 scores into ONE headline number? **That choice is the
most important decision in evaluating LEDGAR.**

## Part 4 — Macro vs Micro vs Weighted F1 (the crucial choice)

Three common ways to average the per-class scores into one number. They answer different
questions:

| Averaging | How it combines classes | Who it favors |
|---|---|---|
| **Macro-F1** | plain average of each class's F1 | **every class counts equally** — rare classes matter as much as common ones |
| **Micro-F1** | pool all predictions together, then one F1 | **every *example* counts equally** — dominated by big classes (≈ accuracy here) |
| **Weighted-F1** | average of per-class F1, weighted by class size | a compromise — big classes pull it up |

Let's compute all three and *see the gap*.

In [ ]:
from sklearn.metrics import f1_score

macro    = f1_score(gold, pred, average="macro",    zero_division=0)
micro    = f1_score(gold, pred, average="micro",    zero_division=0)
weighted = f1_score(gold, pred, average="weighted", zero_division=0)

print(f"Macro-F1    : {macro:.3f}   <- averages classes equally (rare classes count fully)")
print(f"Micro-F1    : {micro:.3f}   <- averages examples (big classes dominate; ~= accuracy)")
print(f"Weighted-F1 : {weighted:.3f}   <- in between")

Now the punchline — run our **lazy "always class 0"** model through the *same* averages and
compare:

In [ ]:
print("Real model  -> macro-F1:", round(f1_score(gold, pred, average="macro", zero_division=0), 3))
print("Lazy model  -> macro-F1:", round(f1_score(gold, dumb, average="macro", zero_division=0), 3))
print()
print("Reminder of their ACCURACIES:")
print("  Real model accuracy:", accuracy_score(gold, pred))
print("  Lazy model accuracy:", accuracy_score(gold, dumb))

Look at what just happened:

| Model | Accuracy | Macro-F1 |
|---|---|---|
| Real (imperfect but tries) | 0.80 | **0.73** |
| Lazy (always class 0) | 0.60 | **0.25** |

- By **accuracy**, the lazy model looks only a bit worse (0.60 vs 0.80).
- By **macro-F1**, the lazy model is *exposed*: **0.25** — because it scores 0 on the two
  classes it never predicts, and macro-F1 refuses to let the one big class hide that.

> **Lesson 2:** For LEDGAR (many classes, imbalanced), report and optimize **macro-F1**. It's
> the metric the LEDGAR / LexGLUE papers use, precisely because it can't be gamed by ignoring
> rare classes. Accuracy and micro-F1 will look nicer but can hide a broken model.

## Part 5 — The full report in one call (what you'll use in practice)

`classification_report` prints per-class precision/recall/F1 **and** the macro/weighted
averages together. This is the single command you'll run to evaluate a real model later.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    gold, pred,
    labels=[0, 1, 2],
    target_names=[names[0], names[1], names[2]],
    zero_division=0,
    digits=3,
))

Reading this report:
- One row **per class** with its precision / recall / f1 / support (how many examples it had).
- `accuracy` — the overall (misleading-on-imbalance) number.
- `macro avg` — **the one you care about for LEDGAR**.
- `weighted avg` — the size-weighted compromise.

Scan the per-class rows for any with low recall or an F1 of 0 — those are the classes your
model is failing, usually the rare ones. That's where fine-tuning improvements should show up.

## Recap — your evaluation playbook for LEDGAR

1. **Never trust accuracy alone** here — imbalance lets a lazy model score high while being
   useless (0.60 accuracy for "always guess the popular class").
2. **Precision** = of what you flagged, how much was right (false-alarm guard).
   **Recall** = of what exists, how much you found (miss guard).
3. **F1** = harmonic mean of the two — high only when *both* are high.
4. **Report macro-F1** as the headline. It averages classes equally, so it can't be gamed by
   ignoring rare classes — which is why the LEDGAR papers use it.
5. **Watch per-class F1** (via `classification_report`) to see *which* classes are failing.

### Why this had to come first
When we fine-tune next, "did it improve?" will mean **"did macro-F1 on the validation set go
up?"** — not "did accuracy go up." You now have the yardstick. *Then* we train.
